# 📝 Text Summarization with PEGASUS

This notebook demonstrates how to **fine-tune** the [PEGASUS](https://huggingface.co/google/pegasus-cnn_dailymail) model on the [SAMSum](https://huggingface.co/datasets/knkarthick/samsum) dataset for **dialogue summarization**.

## What You Will Learn
- How to load and explore a Hugging Face dataset
- How to tokenize text data for a Seq2Seq model
- How to fine-tune a pre-trained PEGASUS model using the `Trainer` API
- How to evaluate summarization quality using ROUGE metrics
- How to save, reload, and run inference with a fine-tuned model

## Dataset
**SAMSum** contains ~16,000 chat-style dialogues with human-written summaries. It is ideal for training models that summarize conversational text.

## Model
**PEGASUS** (Pre-training with Extracted Gap-sentences for Abstractive SUmmarization) is a Google model specifically designed for abstractive text summarization.

---
> **Note:** This notebook is designed to run on **Google Colab with a GPU** (Runtime > Change runtime type > T4 GPU).

## Step 1: Check GPU Availability

Before doing anything else, confirm that a GPU is available. PEGASUS is a large model — training on CPU would take many hours.

In [ ]:
# Check GPU info — make sure you see a T4 or similar GPU listed
# If you see 'No devices were found', go to Runtime > Change runtime type > GPU
!nvidia-smi

## Step 2: Install Required Libraries

We pin `transformers` to version **4.46.0** because `transformers 5.x` removed the `'summarization'` pipeline task string. Using 4.46.0 ensures full compatibility with all the APIs used in this notebook.

In [ ]:
# Install core NLP libraries:
# - transformers: Hugging Face model hub and training utilities
# - sentencepiece: tokenizer backend required by PEGASUS
# - datasets: for loading SAMSum and other Hugging Face datasets
# - sacrebleu: standard machine translation/summarization evaluation metric
# - rouge_score: for computing ROUGE evaluation scores
# - py7zr: needed by some dataset loaders for compressed archives
# - evaluate: new Hugging Face library for computing metrics cleanly
!pip install "transformers[sentencepiece]==4.46.0" datasets sacrebleu rouge_score py7zr evaluate -q

In [ ]:
# Install/upgrade the accelerate library which is required by the Trainer API
# accelerate handles distributed training, mixed precision, and device placement automatically
!pip install --upgrade accelerate -q

## Step 3: Import Libraries

In [ ]:
# Hugging Face libraries
from transformers import pipeline, set_seed          # pipeline: easy inference wrapper; set_seed: for reproducibility
from transformers import AutoModelForSeq2SeqLM       # Auto-loads the correct Seq2Seq architecture (e.g., PEGASUS, BART, T5)
from transformers import AutoTokenizer               # Auto-loads the matching tokenizer for a given model checkpoint
from transformers import DataCollatorForSeq2Seq      # Pads and batches Seq2Seq data dynamically during training
from transformers import TrainingArguments, Trainer  # High-level training API

# Dataset and evaluation libraries
from datasets import load_dataset                    # Load datasets from Hugging Face Hub
import evaluate                                      # Compute standard NLP metrics (ROUGE, BLEU, etc.)

# Standard Python libraries
import pandas as pd                                  # For displaying evaluation results in a readable table
import matplotlib.pyplot as plt                      # For optional data visualization
from tqdm import tqdm                                # Progress bars for loops
import torch                                         # PyTorch: the deep learning backend

# NLTK for sentence tokenization (used during evaluation)
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt")                               # Download the punkt sentence tokenizer model
nltk.download("punkt_tab")                           # Download updated punkt_tab resource (required in newer NLTK)

## Step 4: Hugging Face Login

Log in to the Hugging Face Hub. This is required to download gated models or to push your fine-tuned model to the Hub later.

> Your token is stored **temporarily in memory only** and is never saved to disk.

In [ ]:
from huggingface_hub import notebook_login

# This will display a login widget in Colab.
# Generate your token at: https://huggingface.co/settings/tokens
notebook_login()

## Step 5: Set Device (GPU or CPU)

In [ ]:
# Automatically use GPU (CUDA) if available, otherwise fall back to CPU.
# Training this model on CPU is extremely slow — GPU is strongly recommended.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 6: Load the Pre-trained PEGASUS Model and Tokenizer

We use `google/pegasus-cnn_dailymail` as our starting checkpoint. This is a PEGASUS model already pre-trained on CNN/DailyMail news articles. Fine-tuning it on SAMSum will adapt it to conversational dialogue summarization.

In [ ]:
# The model checkpoint identifier on the Hugging Face Hub
# Using the CNN/DailyMail variant as a starting point (good general-purpose summarizer)
model_ckpt = "google/pegasus-cnn_dailymail"

# Load the tokenizer — this converts raw text into token IDs that the model can process
# AutoTokenizer automatically selects the right tokenizer class for PEGASUS
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")

In [ ]:
# Load the pre-trained PEGASUS model and move it to the GPU
# AutoModelForSeq2SeqLM selects the correct architecture for encoder-decoder models
# .to(device) moves all model weights to GPU memory for faster training
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)
print(f"Model loaded and moved to: {device}")

## Step 7: Load and Explore the SAMSum Dataset

In [ ]:
# Load the SAMSum dataset from the Hugging Face Hub
# SAMSum contains messenger-style dialogues with human-written summaries
# It is split into: train (~14,732 samples), validation (~818), test (~819)
dataset_samsum = load_dataset("knkarthick/samsum")
print(dataset_samsum)

In [ ]:
# Print a summary of each split and show an example from the test set
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features (columns): {dataset_samsum['train'].column_names}")

print("\n" + "="*60)
print("Example from Test Set:")
print("="*60)
print("\nDialogue:")
print(dataset_samsum["test"][1]["dialogue"])
print("\nReference Summary:")
print(dataset_samsum["test"][1]["summary"])

## Step 8: Tokenize the Dataset

Before training, every text sample must be converted into **token IDs** (integers) that the model understands. This process is called tokenization.

- `input_ids`: the encoded dialogue (model input)
- `attention_mask`: tells the model which tokens are real vs. padding
- `labels`: the encoded summary (what the model should learn to generate)

In [ ]:
def convert_examples_to_features(example_batch):
    """
    Tokenizes a batch of dialogue-summary pairs for Seq2Seq training.

    Args:
        example_batch (dict): A batch of examples with 'dialogue' and 'summary' keys.

    Returns:
        dict: Contains 'input_ids', 'attention_mask', and 'labels' tensors.
    """
    # Tokenize the input dialogues
    # max_length=1024: PEGASUS supports up to 1024 tokens for input
    # truncation=True: if the dialogue is longer than 1024 tokens, cut it off
    input_encodings = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True
    )

    # Tokenize the target summaries
    # max_length=128: summaries should be concise (128 tokens is usually enough)
    # truncation=True: cut off any summary longer than 128 tokens
    target_encodings = tokenizer(
        example_batch['summary'],
        max_length=128,
        truncation=True
    )

    return {
        'input_ids'     : input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels'        : target_encodings['input_ids']   # 'labels' is the standard key the Trainer expects
    }

In [ ]:
# Apply the tokenization function to the entire dataset
# batched=True processes multiple samples at once for much faster throughput
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched=True)
print("Tokenized dataset:")
print(dataset_samsum_pt)

In [ ]:
# Verify the tokenized output for the first training example
# input_ids is a list of integers — each integer maps to a word/subword token in the vocabulary
print("Sample input_ids (first 20 tokens):")
print(dataset_samsum_pt["train"]["input_ids"][1][:20])

print("\nSample attention_mask (first 20 values):")
print(dataset_samsum_pt["train"]["attention_mask"][1][:20])
# 1 = real token, 0 = padding token (there is no padding yet since we haven't batched)

## Step 9: Set Up the Data Collator

A **data collator** is responsible for combining individual samples into a batch during training. `DataCollatorForSeq2Seq` handles dynamic padding — it pads all sequences in a batch to the same length, which is more efficient than padding everything to the global maximum.

In [ ]:
from transformers import DataCollatorForSeq2Seq

# DataCollatorForSeq2Seq:
# - Dynamically pads input_ids, attention_mask, and labels to the longest sequence in each batch
# - Replaces padding tokens in 'labels' with -100 so the loss function ignores them
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
print("Data collator ready.")

## Step 10: Configure Training Arguments

These hyperparameters control the training process. Each one is explained below.

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',          # Directory to save model checkpoints and logs
    num_train_epochs=1,                   # Number of full passes over the training data
                                          # (1 epoch here for quick demonstration; use 3-5 for better results)
    warmup_steps=500,                     # Linearly increase learning rate for first 500 steps to stabilize training
    per_device_train_batch_size=1,        # Number of training samples per GPU per step
                                          # (PEGASUS is large; batch_size=1 avoids out-of-memory errors)
    per_device_eval_batch_size=1,         # Same for evaluation
    weight_decay=0.01,                    # L2 regularization to prevent overfitting
    logging_steps=10,                     # Log training loss every 10 steps
    eval_strategy='steps',               # Run evaluation at regular step intervals (not just end of epoch)
    eval_steps=500,                       # Evaluate every 500 training steps
    save_steps=1e6,                       # Save checkpoint every 1,000,000 steps (effectively: only save at the end)
    gradient_accumulation_steps=16        # Accumulate gradients over 16 steps before updating weights.
                                          # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps = 16
                                          # This simulates a larger batch size without needing more GPU memory.
)
print("Training arguments configured.")

## Step 11: Create the Trainer and Start Fine-tuning

> **Important Learning Note:** We are training on the `test` split here **only for demonstration purposes** (it is small and trains quickly). For a properly fine-tuned model, change `train_dataset` to `dataset_samsum_pt["train"]`.

In [ ]:
# Create the Trainer object — this manages the entire training loop for us
trainer = Trainer(
    model=model_pegasus,                             # The PEGASUS model to fine-tune
    args=trainer_args,                               # Training configuration from above
    data_collator=seq2seq_data_collator,             # Handles dynamic padding of batches
    train_dataset=dataset_samsum_pt["test"],         # NOTE: Using test split for demo only!
                                                     # Change to dataset_samsum_pt["train"] for real training
    eval_dataset=dataset_samsum_pt["validation"]     # Used to monitor performance during training
)

In [ ]:
# Start training!
# The Trainer will print loss values every 10 steps and run evaluation every 500 steps
# This may take 20-40 minutes on a T4 GPU even with the small test split
trainer.train()

## Step 12: Evaluate with ROUGE Metrics

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is the standard metric for summarization quality. It measures n-gram overlap between the model's generated summary and the reference human summary.

| Metric | Measures |
|---|---|
| ROUGE-1 | Overlap of individual words (unigrams) |
| ROUGE-2 | Overlap of word pairs (bigrams) |
| ROUGE-L | Longest common subsequence |
| ROUGE-Lsum | ROUGE-L computed per sentence, then averaged |

In [ ]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """
    Splits a list into smaller batches for efficient batch inference.

    Args:
        list_of_elements (list): The full list to split.
        batch_size (int): How many elements per chunk.

    Yields:
        list: A sub-list of length <= batch_size.
    """
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]


def calculate_metric_on_test_ds(
    dataset, metric, model, tokenizer,
    batch_size=16,
    device=device,
    column_text="article",
    column_summary="highlights"
):
    """
    Generates summaries for a dataset in batches and computes ROUGE scores.

    Args:
        dataset: Hugging Face dataset split to evaluate on.
        metric: An `evaluate` metric object (e.g., rouge).
        model: The fine-tuned Seq2Seq model.
        tokenizer: The corresponding tokenizer.
        batch_size (int): How many samples to process at once.
        device (str): 'cuda' or 'cpu'.
        column_text (str): Dataset column containing the input text.
        column_summary (str): Dataset column containing the reference summary.

    Returns:
        dict: ROUGE scores (rouge1, rouge2, rougeL, rougeLsum).
    """
    # Split dataset columns into batches
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches  = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):
        # Tokenize the input batch — padding ensures all sequences in the batch are the same length
        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"             # Return PyTorch tensors
        )

        # Generate summaries using beam search
        # length_penalty > 1.0 encourages longer sequences; < 1.0 encourages shorter ones
        # num_beams=8 means the model explores 8 candidate sequences at each step
        # max_length=128 caps the output length at 128 tokens
        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        # Decode generated token IDs back into human-readable text
        # skip_special_tokens=True removes tokens like <pad>, </s>
        decoded_summaries = [
            tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for s in summaries
        ]

        # Clean up any leftover empty sentence tokens
        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        # Add this batch's predictions and references to the metric accumulator
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # Compute and return the final ROUGE scores across all batches
    score = metric.compute()
    return score

In [ ]:
# Define which ROUGE variants we want to report
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

# Load the ROUGE metric from the evaluate library
rouge_metric = evaluate.load('rouge')
print("ROUGE metric loaded.")

In [ ]:
# Compute ROUGE scores on a small subset of the test set (first 10 samples)
# For a full evaluation, remove the [0:10] slice
score = calculate_metric_on_test_ds(
    dataset=dataset_samsum['test'][0:10],
    metric=rouge_metric,
    model=trainer.model,
    tokenizer=tokenizer,
    batch_size=2,
    column_text='dialogue',
    column_summary='summary'
)

# Display results in a clean DataFrame
rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index=['pegasus-samsum'])

## Step 13: Save the Fine-tuned Model

After training, we save both the model weights and the tokenizer to a local directory. This allows us to reload them later without retraining.

In [ ]:
# Save the fine-tuned model weights and config to a local directory
# This saves: config.json, model weights (.safetensors or .bin), generation_config.json
model_pegasus.save_pretrained("pegasus-samsum-model")
print("Model saved to: pegasus-samsum-model/")

In [ ]:
# Save the tokenizer files alongside the model
# This saves: tokenizer_config.json, special_tokens_map.json, spiece.model (vocabulary)
# Always save the tokenizer with the model — they must match!
tokenizer.save_pretrained("pegasus-samsum-model")
print("Tokenizer saved to: pegasus-samsum-model/")

## Step 14: Load the Saved Model for Inference

We reload the model and tokenizer from disk to verify they saved correctly and to run predictions.

In [ ]:
# Reload the fine-tuned model and tokenizer from the local saved directory
# This is also how you would load the model in a new session or deployment
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained("/content/pegasus-samsum-model")
tokenizer     = AutoTokenizer.from_pretrained("/content/pegasus-samsum-model")

print("Model and tokenizer successfully reloaded from disk.")

## Step 15: Run Inference on a Test Sample

Now we use the fine-tuned model to summarize a real dialogue from the test set and compare it to the human-written reference summary.

In [ ]:
# Generation hyperparameters
# length_penalty=0.8: slight preference for shorter summaries
# num_beams=8: beam search with 8 beams for higher quality output
# max_length=128: maximum number of tokens to generate
gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

# Pick the first dialogue from the test set as our sample
sample_text = dataset_samsum["test"][0]["dialogue"]
reference   = dataset_samsum["test"][0]["summary"]

print("Sample loaded. Ready for inference.")

In [ ]:
# Create a summarization pipeline using our fine-tuned model
# The pipeline wraps tokenization + inference + decoding into a single function call
# device=0 means use GPU (device=0 is the first GPU); use device=-1 for CPU
pipe = pipeline(
    "summarization",
    model=model_pegasus,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)
print("Summarization pipeline ready.")

In [ ]:
# Generate and display the summary
print("Dialogue:")
print("-" * 60)
print(sample_text)

print("\nReference Summary (written by a human):")
print("-" * 60)
print(reference)

print("\nModel Generated Summary:")
print("-" * 60)
# pipe() returns a list of dicts; [0]["summary_text"] extracts the generated text
generated_summary = pipe(sample_text, **gen_kwargs)[0]["summary_text"]
print(generated_summary)

---

## Summary of What We Did

| Step | Action |
|---|---|
| 1 | Verified GPU availability |
| 2-3 | Installed libraries and imported dependencies |
| 4-5 | Logged in to Hugging Face and set compute device |
| 6 | Loaded PEGASUS model and tokenizer |
| 7 | Loaded and explored the SAMSum dataset |
| 8 | Tokenized dialogues and summaries |
| 9 | Set up data collator for dynamic padding |
| 10-11 | Configured training arguments and ran fine-tuning |
| 12 | Evaluated with ROUGE metrics |
| 13 | Saved model and tokenizer to disk |
| 14-15 | Reloaded and ran inference on a test sample |

## Next Steps
- Train on the full `train` split (14,732 samples) for better performance
- Increase `num_train_epochs` to 3-5
- Push the model to the Hugging Face Hub with `model.push_to_hub('your-username/pegasus-samsum')`
- Try other base models like `facebook/bart-large-cnn` or `t5-base`